# Used Car Value Retention
This notebook walks through the project. The main code is kept in short Python files in `src/` so it is easier to read and reuse. The collected CSV files are included. No fresh scraping is needed.

Target: **listing price / original MSRP**.

AI usage: OpenAI Codex generated and ran the project code and drafted the report. The student supplied the requirements and project preferences. Review the code and describe any independent changes before submitting.

## 1. Setup
In Colab, run the next cell to clone the public repository. If you already cloned it, the cell reuses that folder. On your own computer, open this notebook from the repository and install `requirements.txt` in your Python environment first.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/used-car-value-retention")
    if not repo.exists():
        subprocess.run(["git", "clone", "https://github.com/rovinako/used-car-value-retention.git", str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

print("Project folder:", Path.cwd())

## 2. Look at the collected data
These are actual parsed listings, not generated sample data. They include duplicates from overlapping search pages. Each source has its own original column names. The MSRP file includes source URLs for the historical trim prices.

In [ ]:
import pandas as pd
from IPython.display import display, Image

for filename in ["cargurus_listings.csv", "iseecars_listings.csv", "msrp.csv"]:
    raw = pd.read_csv("data/raw/" + filename)
    print(filename, raw.shape)
    display(raw.head(3))

## 3. Clean and combine
Read `src/prepare_data.py` for the cleaning steps. It removes repeated VINs before splitting, excludes ambiguous MSRP matches, and makes age and description features. The script checks the minimum row counts.

In [ ]:
%run src/prepare_data.py
cars = pd.read_csv("data/processed/integrated_cars.csv")
display(cars[["make", "model", "year", "price_usd", "original_msrp", "retention_ratio"]].head())
display(cars[["price_usd", "mileage", "vehicle_age", "retention_ratio"]].describe())
display(pd.read_csv("data/processed/cleaning_steps.csv"))

## 4. Compare the models
Read `src/train_models.py` for the full model code. All three models share an 80/20 split. Preprocessing is fitted on training rows only. Three-fold training cross-validation chooses the model; the test set measures final performance.

Trim and city use one-hot encoding with rare categories grouped together. The two text features detect leather and navigation in listing equipment descriptions.

In [ ]:
%run src/train_models.py
display(pd.read_csv("reports/model_results.csv").round(4))
display(pd.read_csv("reports/feature_importance.csv").round(4))

## 5. Visualize the results
MAE and RMSE are measured in retention-ratio units. An MAE of 0.05 means five percentage points of value retention, not five percent price error. Feature importance describes model reliance, not causation.

In [ ]:
for chart in sorted(Path("reports/figures").glob("*.png")):
    display(Image(filename=str(chart), width=850))

## 6. Build the report and check requirements
The final report is saved in `reports/final_report.pdf`. It covers collection, preprocessing, features, model comparison, depreciation, limitations, and AI usage.

In [ ]:
%run src/make_report.py
%run src/check_project.py
from IPython.display import FileLink
display(FileLink("reports/final_report.pdf"))

## Optional: collect new data
Run `python src/collect_data.py` from the repository folder to collect again. It uses cached successful pages by default. Back up the supplied raw CSV files before a new collection. Websites and listings change, so a fresh run may have different counts or blocked pages. Do not replace missing data with invented listings.